# Template Praktikum — Klasifikasi Teks (scikit-learn)

**Cara pakai: ubah `CFG` di §1, lalu Run All. Selesai.**

Notebook ini bukan bahan bacaan — ini mesin. Semua keputusan praktikum (input apa, case apa,
preprocessing mana, fitur apa, model apa) dikumpulkan jadi satu sel konfigurasi. Sel-sel
selanjutnya membaca `CFG` dan menyesuaikan diri sendiri.

Penjelasan teori setiap pilihan ada di **`cheatsheet_klasifikasi_teks.ipynb`**.
Versi neural ada di **`latihan_pytorch.ipynb`** (config-nya bentuknya sama).

| Bagian | Isi |
|---|---|
| §1 | **SEL CONFIG** — satu-satunya yang perlu kamu sentuh |
| §2 | Menu lengkap: semua nilai yang sah untuk tiap opsi |
| §3 | Mesin: load data (semua bentuk input) |
| §4 | Mesin: preprocessing |
| §5 | Mesin: vectorizer + seleksi fitur |
| §6 | Mesin: model |
| §7 | Mesin: latih + evaluasi |
| §8 | **JALANKAN** |
| §9 | Bandingkan beberapa config sekaligus (untuk tabel di laporan) |
| §10 | Analisis: fitur penting + contoh salah klasifikasi |
| §11 | Simpan model & prediksi data baru |

---
## §1 · SEL CONFIG  ·  ubah di sini saja

In [1]:
CFG = {
    # ---------------------------------------------------------------- 1. INPUT
    "sumber":      "satu_file",              # satu_file | train_test | folder_txt | dataframe
    "path":        "data/sentiment140_sample.csv",   # dipakai kalau sumber = satu_file
    "path_train":  "data/split/train.csv",   # dipakai kalau sumber = train_test
    "path_test":   "data/split/test.csv",
    "text_col":    5,                        # None = deteksi otomatis; di sini kolom ke-5
    "label_col":   0,
    "sep":         None,                     # None = tebak dari ekstensi (.tsv -> tab)
    "header":      None,                     # "infer" (ada header) | None (tanpa header)
    "encoding":    None,                     # None = coba utf-8, latin-1, cp1252
    "label_map":   {0: "negatif", 4: "positif"},     # label angka -> nama kelas
    "sample":      3000,                     # subsample stratified biar cepat; None = semua

    # ---------------------------------------------------------------- 2. CASE
    "case":        "biner",                  # biner | multiclass | multilabel | imbalanced
    "pemisah_label": "|",                    # hanya untuk multilabel
    "bahasa":      "en",                     # en | id

    # ---------------------------------------------------------------- 3. PREPROCESSING (semua opsional)
    "lowercase":         True,
    "mask_entity":       True,               # URL, mention, angka, nominal -> token generik
    "buang_stopword":    True,
    "jaga_negasi":       True,               # jangan buang not/tidak (penting untuk sentimen)
    "normalisasi_slang": False,              # gk -> tidak, yg -> yang  (bahasa id)
    "stemming":          False,              # Porter (en) / Sastrawi (id)
    "lemmatization":     False,              # WordNet (en saja)
    "koreksi_ejaan":     False,              # lambat, berisiko merusak data
    "filter_pos":        False,              # sisakan Noun/Verb/Adj saja (en saja, lambat)

    # ---------------------------------------------------------------- 4. FEATURE EXTRACTION
    "fitur":        "tfidf",                 # boolean | tf | tfidf | hashing
    "ngram":        (1, 2),
    "min_df":       2,
    "max_df":       1.0,
    "max_features": None,
    "sublinear_tf": True,                    # 1+log(tf), hanya untuk tfidf
    "svd":          None,                    # None atau jumlah komponen LSA, mis. 200
    "seleksi":      None,                    # None | chi2 | mutual_info
    "k_fitur":      3000,                    # dipakai kalau seleksi != None

    # ---------------------------------------------------------------- 5. MODEL
    "model":        "logreg",                # nb | bernoulli_nb | logreg | svm | tree | rf | mlp | xgboost
    "seimbangkan":  False,                   # class_weight="balanced" (untuk case imbalanced)
    "tuning":       False,                   # GridSearchCV kecil

    # ---------------------------------------------------------------- 6. EVALUASI
    "test_size":    0.2,                     # diabaikan kalau sumber = train_test
    "cv":           5,                       # 0 = matikan cross-validation
    "random_state": 42,
}

---
## §2 · Menu — semua nilai yang sah

### `sumber` — bentuk input
| Nilai | Kapan | Yang perlu diisi |
|---|---|---|
| `satu_file` | satu CSV/TSV berisi semua data | `path` |
| `train_test` | dosen memberi dua file terpisah | `path_train`, `path_test` |
| `folder_txt` | satu folder per kelas berisi `.txt` | `path` = folder induk |
| `dataframe` | data sudah ada di variabel `DF_MANUAL` | isi `DF_MANUAL` sebelum Run All |

### `case` — jenis task
| Nilai | Artinya | Yang berubah otomatis |
|---|---|---|
| `biner` | 2 kelas (spam/ham) | metrik biasa |
| `multiclass` | >2 kelas, satu label per dokumen | `average="macro"`, confusion matrix k×k |
| `multilabel` | satu dokumen bisa punya beberapa label | `MultiLabelBinarizer` + `OneVsRestClassifier`, metrik micro/macro |
| `imbalanced` | kelas sangat timpang | otomatis menyarankan `seimbangkan=True`, fokus ke f1 kelas minoritas |

### `fitur` — feature extraction (slide 02a hal. 16)
| Nilai | Bobot sel matriks | Cocok untuk |
|---|---|---|
| `boolean` | 0/1 ada-tidaknya term | teks sangat pendek (SMS, judul) |
| `tf` | frekuensi term | dokumen panjang |
| `tfidf` | tf × idf | default paling aman |
| `hashing` | hashing trick, tanpa vocabulary | data sangat besar / streaming |

Tambahan: `svd` = jumlah komponen **LSA** (slide 02b hal. 7–9), `seleksi` = `chi2` atau
`mutual_info` (**MI**, slide 02a hal. 17).

### `model`
| Nilai | Kelas sklearn | Catatan |
|---|---|---|
| `nb` | `MultinomialNB` | baseline wajib, sangat cepat |
| `bernoulli_nb` | `BernoulliNB` | pasangan alami `fitur="boolean"` |
| `logreg` | `LogisticRegression` | default yang solid, koefisiennya bisa dibaca |
| `svm` | `LinearSVC` | sering terbaik untuk teks |
| `tree` | `DecisionTreeClassifier` | bisa digambar, seperti HW-2 |
| `rf` | `RandomForestClassifier` | lebih kuat dari satu pohon |
| `mlp` | `MLPClassifier` | inilah "NN" di slide 02a hal. 18, tanpa PyTorch |
| `xgboost` | `XGBClassifier` | disebut di slide; butuh paket `xgboost` |

### Kombinasi yang tidak boleh
- `nb` / `bernoulli_nb` **tidak menerima nilai negatif** → jangan dipakai bersama `svd`.
- `nb`, `bernoulli_nb`, `mlp`, dan `xgboost` tidak punya `class_weight` → `seimbangkan` diabaikan.
- `svm` tidak punya `predict_proba`.
- `lemmatization`, `filter_pos`, `koreksi_ejaan` hanya untuk `bahasa="en"`.
- `stemming` untuk `bahasa="id"` butuh `pip install Sastrawi`; kalau tidak ada, dilewati otomatis.

Semua pelanggaran di atas **tidak membuat notebook mati** — §3 memvalidasi config, memberi
peringatan, dan memperbaiki sendiri seperlunya.

---
## §3 · Mesin: validasi config + load data

In [2]:
import re, warnings, difflib
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold, GridSearchCV
from sklearn.metrics import (classification_report, confusion_matrix, f1_score,
                             accuracy_score)

warnings.filterwarnings("ignore", category=UserWarning)
DF_MANUAL = None            # diisi sendiri kalau CFG["sumber"] == "dataframe"

PILIHAN = {
    "sumber": {"satu_file", "train_test", "folder_txt", "dataframe"},
    "case":   {"biner", "multiclass", "multilabel", "imbalanced"},
    "bahasa": {"en", "id"},
    "fitur":  {"boolean", "tf", "tfidf", "hashing"},
    "model":  {"nb", "bernoulli_nb", "logreg", "svm", "tree", "rf", "mlp", "xgboost"},
    "seleksi": {None, "chi2", "mutual_info"},
}


def validasi(cfg):
    cfg = dict(cfg)
    catatan = []
    for kunci, sah in PILIHAN.items():
        if cfg.get(kunci) not in sah:
            raise ValueError(f'CFG["{kunci}"] = {cfg.get(kunci)!r} tidak sah. Pilihan: {sorted(map(str, sah))}')

    if cfg["model"] in {"nb", "bernoulli_nb"} and cfg.get("svd"):
        catatan.append("NB tidak menerima nilai negatif dari SVD -> svd dimatikan")
        cfg["svd"] = None
    if cfg["seimbangkan"] and cfg["model"] in {"nb", "bernoulli_nb", "mlp", "xgboost"}:
        catatan.append(f'{cfg["model"]} tidak punya class_weight -> seimbangkan diabaikan')
        cfg["seimbangkan"] = False
    if cfg["bahasa"] == "id" and (cfg["lemmatization"] or cfg["filter_pos"]):
        catatan.append("lemmatization/filter_pos hanya untuk bahasa en -> dimatikan")
        cfg["lemmatization"] = cfg["filter_pos"] = False
    if cfg["case"] == "multilabel" and cfg["model"] in {"tree", "rf"}:
        catatan.append("tree/rf kurang cocok untuk multilabel OvR -> tetap dijalankan, hasil bisa jelek")
    if cfg["fitur"] == "hashing" and cfg["seleksi"] == "chi2":
        catatan.append("hashing bisa menghasilkan nilai negatif -> chi2 dimatikan")
        cfg["seleksi"] = None
    for c in catatan:
        print("  [config disesuaikan]", c)
    return cfg

In [3]:
def _baca_tabel(path, cfg):
    path = Path(path)
    sep = cfg["sep"] or ("\t" if path.suffix.lower() in {".tsv", ".tab"} else ",")
    for enc in ([cfg["encoding"]] if cfg["encoding"] else ["utf-8", "latin-1", "cp1252"]):
        try:
            return pd.read_csv(path, sep=sep, header=cfg["header"], encoding=enc,
                               engine="python", on_bad_lines="skip")
        except (UnicodeDecodeError, UnicodeError):
            continue
    raise ValueError(f"gagal membaca {path} dengan encoding mana pun")


def _pilih_kolom(df, cfg):
    text_col, label_col = cfg["text_col"], cfg["label_col"]
    if text_col is None:
        kand = [c for c in df.columns if df[c].map(lambda v: isinstance(v, str)).mean() > 0.5]
        if not kand:
            raise ValueError('kolom teks tidak terdeteksi, isi CFG["text_col"]')
        text_col = max(kand, key=lambda c: df[c].astype(str).str.len().mean())
    if label_col is None:
        batas = 200 if cfg["case"] == "multilabel" else 20
        kand = [(df[c].nunique(dropna=True), c) for c in df.columns
                if c != text_col and 2 <= df[c].nunique(dropna=True) <= batas]
        if not kand:
            raise ValueError('kolom label tidak terdeteksi, isi CFG["label_col"]')
        label_col = min(kand)[1]
    return text_col, label_col


def _rapikan(df, cfg):
    df = df.copy()
    df["text"] = df["text"].astype(str).str.strip()
    if cfg["case"] != "multilabel":
        df["label"] = df["label"].apply(lambda v: v.strip().lower() if isinstance(v, str) else v)
        if cfg["label_map"]:
            df["label"] = df["label"].map(cfg["label_map"]).fillna(df["label"])
    df = df[df["text"].str.len() >= 3]
    df = df[~df["text"].str.lower().isin({"nan", "none", "na", "-"})]
    return df.dropna(subset=["text", "label"]).drop_duplicates(subset=["text"]).reset_index(drop=True)


def muat(cfg):
    if cfg["sumber"] == "dataframe":
        if DF_MANUAL is None:
            raise ValueError("CFG['sumber']='dataframe' tapi DF_MANUAL masih None")
        df = DF_MANUAL.copy()
        tc, lc = _pilih_kolom(df, cfg)
        df = df.rename(columns={tc: "text", lc: "label"})[["text", "label"]]
        return _rapikan(df, cfg), None

    if cfg["sumber"] == "folder_txt":
        from sklearn.datasets import load_files
        b = load_files(cfg["path"], encoding="utf-8", decode_error="replace")
        df = pd.DataFrame({"text": b.data, "label": [b.target_names[i] for i in b.target]})
        return _rapikan(df, cfg), None

    if cfg["sumber"] == "train_test":
        hasil = []
        for p in (cfg["path_train"], cfg["path_test"]):
            d = _baca_tabel(p, cfg)
            tc, lc = _pilih_kolom(d, cfg)
            d = d.rename(columns={tc: "text", lc: "label"})[["text", "label"]]
            hasil.append(_rapikan(d, cfg))
        tr, te = hasil
        bocor = set(tr["text"]) & set(te["text"])
        if bocor:
            print(f"  [peringatan] {len(bocor)} dokumen bocor train<->test, dibuang dari test")
            te = te[~te["text"].isin(bocor)].reset_index(drop=True)
        return tr, te

    d = _baca_tabel(cfg["path"], cfg)
    tc, lc = _pilih_kolom(d, cfg)
    d = d.rename(columns={tc: "text", lc: "label"})[["text", "label"]]
    d = _rapikan(d, cfg)
    if cfg["sample"] and cfg["sample"] < len(d):
        strat = d["label"] if cfg["case"] != "multilabel" else None
        d, _ = train_test_split(d, train_size=cfg["sample"], stratify=strat,
                                random_state=cfg["random_state"])
        d = d.reset_index(drop=True)
    return d, None

---
## §4 · Mesin: preprocessing

Semua sakelar `True/False` di `CFG` bagian 3 dibaca di sini. Kalau semuanya `False`, teks
diteruskan apa adanya dan vectorizer tetap bekerja — bagian ini memang opsional seluruhnya.

In [4]:
STOP_EN = {"i","me","my","we","our","you","your","he","she","it","they","them","this","that",
           "these","those","am","is","are","was","were","be","been","being","have","has","had",
           "do","does","did","a","an","the","and","but","if","or","because","as","of","at","by",
           "for","with","to","from","in","out","on","off","then","so","than","too","very","will",
           "just","now","s","t","can","should","there","here","what","when","how","all","any"}
STOP_ID = {"yang","dan","di","ke","dari","ini","itu","untuk","dengan","pada","adalah","ada",
           "saya","kamu","dia","kami","kita","mereka","akan","sudah","telah","juga","atau",
           "karena","agar","saja","oleh","sebagai","dalam","para","nya","banget","sekali",
           "sangat","pokoknya","tetapi","tapi","lagi","bisa","masih","sih"}
NEGASI_EN = {"no","not","never","nor","cannot","dont","didnt","isnt","wasnt","wont"}
NEGASI_ID = {"tidak","bukan","tanpa","jangan","belum","kurang"}
SLANG_ID = {"gk":"tidak","ga":"tidak","gak":"tidak","nggak":"tidak","engga":"tidak","yg":"yang",
            "dgn":"dengan","tdk":"tidak","sy":"saya","bgt":"banget","udh":"sudah","blm":"belum",
            "trs":"terus","krn":"karena","aja":"saja","gitu":"begitu","udah":"sudah"}

NLTK_OK = False
try:
    import nltk
    def _punya(p):
        for k in (p, p + ".zip"):
            try:
                nltk.data.find(k); return True
            except LookupError:
                continue
        return False
    for pkg, p in [("punkt","tokenizers/punkt"), ("punkt_tab","tokenizers/punkt_tab"),
                   ("stopwords","corpora/stopwords"), ("wordnet","corpora/wordnet"),
                   ("omw-1.4","corpora/omw-1.4")]:
        if not _punya(p):
            try: nltk.download(pkg, quiet=True)
            except Exception: pass
    from nltk.corpus import stopwords as _sw
    from nltk.stem import PorterStemmer, WordNetLemmatizer
    STOP_EN = set(_sw.words("english"))
    _stemmer, _lemma = PorterStemmer(), WordNetLemmatizer()
    NLTK_OK = True
except Exception as e:
    print("  [info] NLTK tidak siap:", type(e).__name__, "-> pakai daftar stopword bawaan")

try:
    from Sastrawi.Stemmer.StemmerFactory import StemmerFactory
    _stem_id = StemmerFactory().create_stemmer()
    SASTRAWI_OK = True
except Exception:
    _stem_id, SASTRAWI_OK = None, False


def mask_entities(t):
    t = re.sub(r"[\w.+-]+@[\w-]+\.[\w.]+", " emailtoken ", t)
    t = re.sub(r"http\S+|www\.\S+|\b\S+\.(?:com|org|net|ly|id|co)\S*", " urltoken ", t)
    t = re.sub(r"@\w+", " usertoken ", t)
    t = re.sub(r"\b\d{7,}\b", " phonetoken ", t)
    t = re.sub(r"[$£€]\s?\d[\d,.]*", " moneytoken ", t)
    t = re.sub(r"\b\d+\b", " numtoken ", t)
    return t

In [5]:
def buat_preprocessor(cfg):
    """Kembalikan fungsi teks -> teks bersih, sesuai sakelar di CFG."""
    stop  = (STOP_EN if cfg["bahasa"] == "en" else STOP_ID)
    negasi = (NEGASI_EN if cfg["bahasa"] == "en" else NEGASI_ID)
    if cfg["jaga_negasi"]:
        stop = stop - negasi
    kosakata = {"__belum_dibangun__"}

    def bangun_kosakata(daftar):                     # untuk koreksi ejaan
        nonlocal kosakata
        c = Counter(w for t in daftar for w in re.findall(r"[a-z]+", str(t).lower()))
        kosakata = {w for w, n in c.items() if n >= 2}

    def proses(teks):
        t = str(teks)
        if cfg["lowercase"]:
            t = t.lower()
        if cfg["mask_entity"]:
            t = mask_entities(t)
        tokens = re.findall(r"[a-zA-Z]+", t)
        if cfg["normalisasi_slang"]:
            tokens = [SLANG_ID.get(w, w) for w in tokens]
        if cfg["koreksi_ejaan"] and len(kosakata) > 1:
            tokens = [w if (w in kosakata or len(w) <= 3)
                      else (difflib.get_close_matches(w, kosakata, n=1, cutoff=0.85) or [w])[0]
                      for w in tokens]
        if cfg["filter_pos"] and NLTK_OK:
            try:
                tokens = [w for w, tag in nltk.pos_tag(tokens) if tag[0] in ("N", "V", "J")]
            except Exception:
                pass
        if cfg["buang_stopword"]:
            tokens = [w for w in tokens if w not in stop]
        if cfg["stemming"]:
            if cfg["bahasa"] == "en" and NLTK_OK:
                tokens = [_stemmer.stem(w) for w in tokens]
            elif cfg["bahasa"] == "id" and SASTRAWI_OK:
                tokens = [_stem_id.stem(w) for w in tokens]
        if cfg["lemmatization"] and NLTK_OK and cfg["bahasa"] == "en":
            tokens = [_lemma.lemmatize(w, pos="v") for w in tokens]
        return " ".join(tokens) if tokens else "kosongtoken"

    proses.bangun_kosakata = bangun_kosakata
    return proses

---
## §5 · Mesin: vectorizer + reduksi/seleksi fitur

In [6]:
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer, HashingVectorizer
from sklearn.feature_selection import SelectKBest, chi2, mutual_info_classif
from sklearn.decomposition import TruncatedSVD
from sklearn.pipeline import Pipeline


def buat_vectorizer(cfg):
    umum = dict(ngram_range=tuple(cfg["ngram"]), lowercase=False)   # lowercase sudah di preprocessing
    if cfg["fitur"] == "hashing":
        return HashingVectorizer(n_features=2 ** 18, alternate_sign=False, **umum)
    umum.update(min_df=cfg["min_df"], max_df=cfg["max_df"], max_features=cfg["max_features"])
    if cfg["fitur"] == "boolean":
        return CountVectorizer(binary=True, **umum)
    if cfg["fitur"] == "tf":
        return CountVectorizer(**umum)
    return TfidfVectorizer(sublinear_tf=cfg["sublinear_tf"], **umum)


def langkah_fitur(cfg):
    langkah = [("vect", buat_vectorizer(cfg))]
    if cfg["seleksi"] == "chi2":
        langkah.append(("pilih", SelectKBest(chi2, k=cfg["k_fitur"])))
    elif cfg["seleksi"] == "mutual_info":
        skor = lambda X, y: mutual_info_classif(X, y, random_state=cfg["random_state"])
        langkah.append(("pilih", SelectKBest(skor, k=cfg["k_fitur"])))
    if cfg["svd"]:
        langkah.append(("svd", TruncatedSVD(n_components=cfg["svd"],
                                            random_state=cfg["random_state"])))
    return langkah

---
## §6 · Mesin: model

In [7]:
from sklearn.naive_bayes import MultinomialNB, BernoulliNB
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.multiclass import OneVsRestClassifier
from sklearn.base import BaseEstimator, ClassifierMixin
from sklearn.preprocessing import LabelEncoder


class XGBTeks(BaseEstimator, ClassifierMixin):
    """XGBoost hanya menerima label numerik 0..k-1 -> dibungkus LabelEncoder."""

    def __init__(self, **kw):
        self.kw = kw

    def fit(self, X, y):
        from xgboost import XGBClassifier
        self.le_ = LabelEncoder().fit(y)
        self.clf_ = XGBClassifier(**self.kw).fit(X, self.le_.transform(y))
        self.classes_ = self.le_.classes_
        return self

    def predict(self, X):
        return self.le_.inverse_transform(self.clf_.predict(X))


def buat_model(cfg):
    bobot = "balanced" if cfg["seimbangkan"] else None
    rs = cfg["random_state"]
    if cfg["model"] == "nb":
        m = MultinomialNB()
    elif cfg["model"] == "bernoulli_nb":
        m = BernoulliNB()
    elif cfg["model"] == "logreg":
        m = LogisticRegression(max_iter=2000, class_weight=bobot, random_state=rs)
    elif cfg["model"] == "svm":
        m = LinearSVC(class_weight=bobot, random_state=rs)
    elif cfg["model"] == "tree":
        m = DecisionTreeClassifier(max_depth=8, class_weight=bobot, random_state=rs)
    elif cfg["model"] == "rf":
        m = RandomForestClassifier(n_estimators=200, class_weight=bobot,
                                   random_state=rs, n_jobs=-1)
    elif cfg["model"] == "mlp":
        m = MLPClassifier(hidden_layer_sizes=(128,), max_iter=300, random_state=rs)
    elif cfg["model"] == "xgboost":
        m = XGBTeks(n_estimators=300, max_depth=6, learning_rate=0.2,
                    tree_method="hist", random_state=rs, verbosity=0)
    if cfg["case"] == "multilabel":
        m = OneVsRestClassifier(m)
    return m


GRID = {
    "nb":           {"clf__alpha": [0.1, 0.5, 1.0]},
    "bernoulli_nb": {"clf__alpha": [0.1, 0.5, 1.0]},
    "logreg":       {"clf__C": [0.5, 1.0, 5.0]},
    "svm":          {"clf__C": [0.1, 0.5, 1.0]},
    "tree":         {"clf__max_depth": [5, 10, None]},
    "rf":           {"clf__n_estimators": [100, 300]},
    "mlp":          {"clf__alpha": [1e-4, 1e-3]},
    "xgboost":      {"clf__max_depth": [4, 6]},
}

---
## §7 · Mesin: latih + evaluasi

In [8]:
def siapkan_label(df_tr, df_te, cfg):
    """Kembalikan y_train, y_test, nama_kelas, dan encoder (untuk multilabel)."""
    if cfg["case"] != "multilabel":
        # np.asarray(dtype=object), bukan .values -- pandas 3 mengembalikan ArrowStringArray
        # yang bikin cross_val_score error saat mengindeks fold
        return (np.asarray(df_tr["label"], dtype=object),
                np.asarray(df_te["label"], dtype=object),
                sorted(set(map(str, df_tr["label"]))), None)
    from sklearn.preprocessing import MultiLabelBinarizer
    pisah = lambda s: [x.strip() for x in str(s).split(cfg["pemisah_label"]) if x.strip()]
    mlb = MultiLabelBinarizer()
    y_tr = mlb.fit_transform(df_tr["label"].map(pisah))
    y_te = mlb.transform(df_te["label"].map(pisah))
    return y_tr, y_te, list(mlb.classes_), mlb


def jalankan(cfg, verbose=True):
    cfg = validasi(cfg)
    p = print if verbose else (lambda *a, **k: None)

    # --- data
    df_tr, df_te = muat(cfg)
    if df_te is None:
        strat = df_tr["label"] if cfg["case"] != "multilabel" else None
        df_tr, df_te = train_test_split(df_tr, test_size=cfg["test_size"], stratify=strat,
                                        random_state=cfg["random_state"])
        df_tr, df_te = df_tr.reset_index(drop=True), df_te.reset_index(drop=True)
    p(f"data      : train {len(df_tr)} | test {len(df_te)}")
    if cfg["case"] != "multilabel":
        dist = df_tr["label"].value_counts()
        p(f"distribusi: {dist.to_dict()}  (rasio timpang {dist.max()/dist.min():.1f}x)")

    # --- preprocessing
    pre = buat_preprocessor(cfg)
    if cfg["koreksi_ejaan"]:
        pre.bangun_kosakata(df_tr["text"])          # kosakata HANYA dari train
    # pakai Series, bukan list -- cross_val_score mengindeks dengan array numpy
    Xtr = pd.Series([pre(t) for t in df_tr["text"]])
    Xte = pd.Series([pre(t) for t in df_te["text"]])
    p(f"contoh    : {df_tr['text'].iloc[0][:60]}")
    p(f"  -> bersih: {Xtr[0][:60]}")

    # --- label
    ytr, yte, kelas, mlb = siapkan_label(df_tr, df_te, cfg)

    # --- pipeline
    pipe = Pipeline(langkah_fitur(cfg) + [("clf", buat_model(cfg))])
    rata = "micro" if cfg["case"] == "multilabel" else "macro"

    if cfg["tuning"]:
        grid = GRID[cfg["model"]]
        if cfg["case"] == "multilabel":
            grid = {k.replace("clf__", "clf__estimator__"): v for k, v in grid.items()}
        gs = GridSearchCV(pipe, grid, cv=3, scoring=f"f1_{rata}", n_jobs=-1).fit(Xtr, ytr)
        pipe = gs.best_estimator_
        p(f"tuning    : {gs.best_params_} -> cv f1 {gs.best_score_:.3f}")
    else:
        pipe.fit(Xtr, ytr)

    # --- evaluasi
    pred = pipe.predict(Xte)
    f1 = f1_score(yte, pred, average=rata, zero_division=0)
    akurasi = accuracy_score(yte, pred)
    n_fitur = getattr(pipe.named_steps["vect"], "vocabulary_", None)
    p(f"fitur     : {len(n_fitur) if n_fitur else 'hashing/'+str(2**18)} kolom")
    p(f"\nakurasi {akurasi:.3f} | f1_{rata} {f1:.3f}\n")
    p(classification_report(yte, pred, target_names=[str(k) for k in kelas], zero_division=0))

    if cfg["case"] != "multilabel":
        cm = confusion_matrix(yte, pred, labels=kelas)
        p(pd.DataFrame(cm, index=["true_" + str(k) for k in kelas],
                       columns=["pred_" + str(k) for k in kelas]).to_string())

    skor_cv = None
    if cfg["cv"] and cfg["case"] != "multilabel":
        cv = StratifiedKFold(cfg["cv"], shuffle=True, random_state=cfg["random_state"])
        skor = cross_val_score(pipe, Xtr, ytr, cv=cv, scoring=f"f1_{rata}", n_jobs=-1)
        skor_cv = (skor.mean(), skor.std())
        p(f"\nCV {cfg['cv']}-fold di train: {skor.mean():.3f} +/- {skor.std():.3f}")

    return {"pipeline": pipe, "cfg": cfg, "akurasi": akurasi, "f1": f1, "cv": skor_cv,
            "kelas": kelas, "mlb": mlb, "pred": pred, "y_test": yte,
            "df_train": df_tr, "df_test": df_te, "X_train": Xtr, "X_test": Xte, "pre": pre}

---
## §8 · JALANKAN

In [9]:
hasil = jalankan(CFG)

data      : train 2400 | test 600
distribusi: {'positif': 1200, 'negatif': 1200}  (rasio timpang 1.0x)
contoh    : Chris is still standing - but he's working on it
  -> bersih: chris still standing working
fitur     : 2568 kolom

akurasi 0.690 | f1_macro 0.690

              precision    recall  f1-score   support

     negatif       0.70      0.66      0.68       300
     positif       0.68      0.72      0.70       300

    accuracy                           0.69       600
   macro avg       0.69      0.69      0.69       600
weighted avg       0.69      0.69      0.69       600

              pred_negatif  pred_positif
true_negatif           198           102
true_positif            84           216



CV 5-fold di train: 0.685 +/- 0.012


---
## §9 · Bandingkan beberapa config sekaligus

Ini yang biasanya diminta di laporan: tabel "pengaruh tiap pilihan terhadap skor". Ubah daftar
`VARIAN` di bawah — tiap entri hanya menimpa sebagian `CFG`, sisanya ikut config utama.

In [10]:
VARIAN = [
    ("baseline: tfidf + logreg",     {}),
    ("tanpa preprocessing",          {"lowercase": False, "mask_entity": False,
                                      "buang_stopword": False}),
    ("unigram saja",                 {"ngram": (1, 1)}),
    ("fitur boolean + BernoulliNB",  {"fitur": "boolean", "model": "bernoulli_nb"}),
    ("naive bayes",                  {"model": "nb"}),
    ("linear svm",                   {"model": "svm"}),
    ("decision tree",                {"model": "tree"}),
    ("MLP (neural, sklearn)",        {"model": "mlp"}),
    ("+ stemming",                   {"stemming": True}),
    ("+ seleksi chi2 (1000 fitur)",  {"seleksi": "chi2", "k_fitur": 1000}),
    ("+ LSA 200 komponen",           {"svd": 200}),
]

baris = []
for nama, ubah in VARIAN:
    cfg = {**CFG, **ubah, "cv": 0}
    try:
        r = jalankan(cfg, verbose=False)
        baris.append((nama, round(r["akurasi"], 3), round(r["f1"], 3)))
    except Exception as e:
        baris.append((nama, "gagal", f"{type(e).__name__}: {str(e)[:40]}"))

tabel = pd.DataFrame(baris, columns=["konfigurasi", "akurasi", "f1"])
print(tabel.to_string(index=False))

                konfigurasi  akurasi    f1
   baseline: tfidf + logreg    0.690 0.690
        tanpa preprocessing    0.682 0.682
               unigram saja    0.697 0.697
fitur boolean + BernoulliNB    0.683 0.682
                naive bayes    0.677 0.676
                 linear svm    0.660 0.659
              decision tree    0.613 0.612
      MLP (neural, sklearn)    0.635 0.634
                 + stemming    0.693 0.693
+ seleksi chi2 (1000 fitur)    0.693 0.693
         + LSA 200 komponen    0.693 0.693


---
## §10 · Analisis: fitur penting + kesalahan model

In [11]:
def fitur_teratas(hasil, n=12):
    pipe, kelas = hasil["pipeline"], hasil["kelas"]
    vect = pipe.named_steps["vect"]
    if not hasattr(vect, "get_feature_names_out"):
        print("(HashingVectorizer tidak menyimpan nama fitur)"); return
    if "svd" in pipe.named_steps or "pilih" in pipe.named_steps:
        print("(ada langkah SVD/seleksi -> nama fitur tidak sejajar dengan koefisien)"); return
    nama = np.array(vect.get_feature_names_out())
    clf = pipe.named_steps["clf"]
    if hasattr(clf, "coef_") and clf.coef_.shape[0] == 1:
        s = clf.coef_[0]
        print(f"[{kelas[0]}] <-- ", ", ".join(nama[np.argsort(s)[:n]]))
        print(f"[{kelas[1]}] --> ", ", ".join(nama[np.argsort(s)[-n:][::-1]]))
    elif hasattr(clf, "coef_"):
        for i, k in enumerate(kelas):
            print(f"[{k}] ", ", ".join(nama[np.argsort(clf.coef_[i])[-n:][::-1]]))
    elif hasattr(clf, "feature_log_prob_"):
        for i, k in enumerate(kelas):
            khas = clf.feature_log_prob_[i] - clf.feature_log_prob_.mean(axis=0)
            print(f"[{k}] ", ", ".join(nama[np.argsort(khas)[-n:][::-1]]))
    elif hasattr(clf, "feature_importances_"):
        idx = np.argsort(clf.feature_importances_)[-n:][::-1]
        print("fitur terpenting:", ", ".join(nama[idx]))
    else:
        print("(model ini tidak mengekspos bobot fitur)")


fitur_teratas(hasil)

[negatif] <--  sorry, sad, miss, no, not, hate, work, sucks, wish, sick, feel, wanna
[positif] -->  thanks, love, happy, thank, lol, follow, yay, awesome, usertoken, urltoken, great, good


In [12]:
if hasil["cfg"]["case"] != "multilabel":
    salah = pd.DataFrame({"text": hasil["df_test"]["text"].values,
                          "aktual": hasil["y_test"], "prediksi": hasil["pred"]})
    salah = salah[salah.aktual != salah.prediksi]
    print(f"salah klasifikasi: {len(salah)} dari {len(hasil['y_test'])}")
    for _, r in salah.head(8).iterrows():
        print(f"  [aktual={r.aktual} prediksi={r.prediksi}] {r.text[:75]}")
else:
    print("multilabel: lihat classification_report per label di atas")

salah klasifikasi: 186 dari 600
  [aktual=positif prediksi=negatif] I suck at golf! =( but at least I can swing
  [aktual=negatif prediksi=positif] @anitaerikson hehe, I remember that ANZ rainbow  #EFTPOS
  [aktual=positif prediksi=negatif] My phone SIM card is broken. But I can send and receive texts. So text if y
  [aktual=negatif prediksi=positif] Woo hoo falling asleep when everyone else is waking up...oh wait...
  [aktual=negatif prediksi=positif] I want Ray's mask.  it's so creative and beautiful!
  [aktual=positif prediksi=negatif] wow ! I miss a WHOLE lot on here  weekend was really good  movie was amazzi
  [aktual=negatif prediksi=positif] At work &amp; busy.  Hating that I can't check on my boys since Matt's phon
  [aktual=positif prediksi=negatif] Hello peeps I'm hungry kind of I hope I fill better tomorrow because I wann


---
## §11 · Simpan model & prediksi teks baru

In [13]:
import joblib

joblib.dump({"pipeline": hasil["pipeline"], "cfg": hasil["cfg"]}, "model_latihan.joblib")
muat_lagi = joblib.load("model_latihan.joblib")


def prediksi_teks(daftar_teks, bundel=None):
    bundel = bundel or muat_lagi
    pre = buat_preprocessor(bundel["cfg"])
    bersih = [pre(t) for t in daftar_teks]
    keluar = bundel["pipeline"].predict(bersih)
    return list(zip(daftar_teks, keluar))


contoh_baru = ["absolutely love this, best day ever thanks",
               "worst service ever, i hate it, so disappointed"]
for teks, label in prediksi_teks(contoh_baru):
    print(f"  {str(label):10s} <- {teks[:60]}")

  positif    <- absolutely love this, best day ever thanks
  negatif    <- worst service ever, i hate it, so disappointed


---
## Resep cepat per skenario

| Kalau dosen bilang… | Ubah di CFG |
|---|---|
| "pakai dua file train dan test ini" | `"sumber": "train_test"` + isi `path_train`/`path_test` |
| "datanya bahasa Indonesia" | `"bahasa": "id"`, `"normalisasi_slang": True`, `"stemming": True` |
| "klasifikasi topik, 4 kelas" | `"case": "multiclass"`, `"path": "data/berita_topik.tsv"` |
| "satu berita bisa punya beberapa topik" | `"case": "multilabel"`, `"path": "data/berita_multilabel.csv"`, `"label_col": "labels"` |
| "datanya timpang, kelas minoritas penting" | `"case": "imbalanced"`, `"seimbangkan": True`, `"model": "logreg"` |
| "bandingkan Boolean, TF, dan TF-IDF" | jalankan §9 dengan varian `{"fitur": ...}` |
| "coba pakai neural network" | `"model": "mlp"` — atau pindah ke `latihan_pytorch.ipynb` |
| "kurangi jumlah fitur" | `"seleksi": "chi2"`, `"k_fitur": 1000` — atau `"svd": 200` |
| "labelnya angka 0 dan 4" | `"label_map": {0: "negatif", 4: "positif"}` |
| "file CSV-nya tidak ada headernya" | `"header": None`, lalu `"text_col": 5`, `"label_col": 0` |